# AI工学101 — 第19回

## ハイパーパラメータ探索：モデルの「設定」をデータから選ぶ

よし、ここまで来たぞ💪
第18回では**交差検証（Cross Validation）**を使って、「このモデルはたまたま良かっただけじゃないか？」を確認しました。

今回は、その一歩先。

> **モデルの設定をどう選ぶか**

を扱います。

機械学習では、モデルの中には**学習によって決まる値**と、**人間があらかじめ設定する値**があります。

この区別は、これからPyTorchまで進むうえでかなり重要です。

---

# 🎯 今日のゴール

今日は以下を身につけます。

* パラメータとハイパーパラメータの違いを理解する
* `C` などのハイパーパラメータが何を意味するか理解する
* `GridSearchCV` を使える
* 交差検証とハイパーパラメータ探索を組み合わせる
* **テストデータを最終評価用に温存する**という考え方を理解する

---

# 📖 講義：約20分

## 1. パラメータとハイパーパラメータ

例えばロジスティック回帰では、

[
z = Wx+b
]

というモデルを学習します。

この

```text
W
b
```

はデータから `fit()` によって決まります。

これは**モデルパラメータ**です。

一方、

```python
LogisticRegression(C=1.0)
```

の

```text
C
```

は、人間が設定します。

これが**ハイパーパラメータ**です。

---

## 2. なぜ設定を変えるのか？

例えば、

```python
LogisticRegression(C=0.01)
```

と

```python
LogisticRegression(C=100)
```

では、モデルの振る舞いが変わります。

でも、

> 「Cは1.0が絶対に正しい！」

という理論があるとは限りません。

そこで、

```text
C = 0.01
C = 0.1
C = 1
C = 10
C = 100
```

と試して、

**交差検証で最も良かった設定を選ぶ**

という方法があります。

---

# 🧠 3. Grid Search

これを自動化するのが

> `GridSearchCV`

です。

イメージは、

```text
C=0.01 ─┐
C=0.1  ─┤
C=1    ─┼→ Cross Validation → 比較
C=10   ─┤
C=100  ─┘
```

です。

つまり、

> **ハイパーパラメータ候補を総当たりして、交差検証で比較する**

方法です。

---

# 💻 実習1：データを用意する

今回は少しまともな分類データを使います。

```python
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target
```

確認。

```python
print(X.shape)
print(y.shape)
```

出力は、

```text
(150, 4)
(150,)
```

です。

つまり、

```text
150サンプル
4特徴量
```

です。

---

# 💻 実習2：まずtrain/testを分ける

ここが今日の超重要ポイント。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

今回は、

```python
stratify=y
```

を指定しています。

---

## `stratify` とは？

Irisは3クラスあります。

```text
class 0
class 1
class 2
```

これらの割合をなるべく維持したまま、

```text
train
test
```

に分割します。

分類問題では便利な指定です。

---

# 💻 実習3：Pipelineを作る

前回までの知識を全部つなげます。

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
```

Pipeline。

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])
```

ここではまだ探索しません。

まず、

```python
pipe.fit(X_train, y_train)
```

で普通に学習できます。

---

# 💻 実習4：普通に評価する

```python
from sklearn.metrics import accuracy_score

pred = pipe.predict(X_test)

print(
    accuracy_score(
        y_test,
        pred
    )
)
```

これがベースラインです。

---

# 💻 実習5：GridSearchCV

読み込みます。

```python
from sklearn.model_selection import GridSearchCV
```

探索する値を作ります。

```python
param_grid = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}
```

ここで、

```text
model__C
```

となっているのがポイント。

Pipelineで

```python
("model", LogisticRegression())
```

と名前を付けたので、

```text
model__C
```

で

「modelステップのC」

を指定します。

---

# 💻 実習6：探索開始

```python
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="accuracy"
)
```

学習。

```python
grid.fit(
    X_train,
    y_train
)
```

これで、

```text
C=0.01
 ↓
5-fold CV

C=0.1
 ↓
5-fold CV

C=1
 ↓
5-fold CV

...
```

を自動的に行います。

---

# 💻 実習7：最適な設定を見る

```python
print(
    grid.best_params_
)
```

例えば、

```python
{
    "model__C": 10
}
```

のような結果が出ます。

これが、

> **交差検証で最も良かったハイパーパラメータ**

です。

---

# 💻 実習8：最高スコアを見る

```python
print(
    grid.best_score_
)
```

これは、

**trainデータ内部で行った5-fold CVの最高平均スコア**

です。

ここで注意。

これはまだ、

**最終的なテストスコアではありません。**

---

# 💻 実習9：最終評価

最適なモデルは、

```python
grid.best_estimator_
```

で取得できます。

テストデータで評価。

```python
best_model = grid.best_estimator_

test_pred = best_model.predict(
    X_test
)

test_acc = accuracy_score(
    y_test,
    test_pred
)

print(test_acc)
```

これが、

> **探索を終えた後の最終的な未知データ評価**

です。

---

# 🧠 ここ、めちゃくちゃ重要

今回の流れは、

```text
全データ
   ↓
train / test 分割
   ↓
   ┌───────────────┐
   │     train     │
   │               │
   │ GridSearchCV  │
   │      ↓        │
   │ Cross Valid.  │
   │      ↓        │
   │ 最適な設定決定 │
   └───────────────┘
           ↓
     最終モデル
           ↓
        test
           ↓
      最終評価
```

です。

つまり、

**テストデータをハイパーパラメータ探索に使ってはいけません。**

もし、

```text
testを見る
 ↓
Cを変更
 ↓
testを見る
 ↓
またCを変更
```

をやったら、

そのテストデータに合わせてしまいます。

これは評価の独立性を壊します。

---

# 💻 実習10：探索結果を確認する

GridSearchCVには、

探索した結果が入っています。

```python
results = grid.cv_results_
```

例えば、

```python
print(
    results["params"]
)
```

さらに、

```python
print(
    results["mean_test_score"]
)
```

で、

各設定の平均CVスコアを確認できます。

---

## 対応させて見る

```python
for params, score in zip(
    results["params"],
    results["mean_test_score"]
):
    print(
        params,
        score
    )
```

すると、

```text
{'model__C': 0.01} 0.91
{'model__C': 0.1}  0.94
{'model__C': 1}    0.95
{'model__C': 10}   0.96
{'model__C': 100}  0.95
```

のように、

**設定と性能の関係**

を見ることができます。

これはかなり重要な「実験を見る目」です。

---

# 📖 実験科学として見ると

ここまでの機械学習は、

単に

```python
model.fit()
```

しているわけではありません。

実際には、

```text
仮説
 ↓
条件を設定
 ↓
実験
 ↓
評価
 ↓
比較
 ↓
次の仮説
```

という、

**実験科学に近いループ**

になっています。

---

# ✍️ 演習

今日の `iris` データを使います。

```python
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target
```

---

## 問1

`train_test_split()` を使って、

```text
train 80%

test 20%
```

に分割してください。

`stratify=y` も指定します。

---

## 問2

次のPipelineを作ってください。

```text
StandardScaler
↓
LogisticRegression
```

---

## 問3

`C` を

```text
0.001
0.01
0.1
1
10
100
```

から探索してください。

---

## 問4

5-fold Cross Validationで、

最適な `C` を探してください。

---

## 問5

```python
best_params_
```

と

```python
best_score_
```

を表示してください。

---

## 問6

最適モデルをtestデータで評価してください。

---

## 問7（ボス戦👾）

探索した全候補について、

```text
C

CV平均Accuracy
```

を一覧表示してください。

さらに、

> **Cを大きくするとAccuracyはどう変化したか？**

を自分の結果から説明してください。

ここでは「正解」を暗記する必要はありません。

**実験結果から傾向を読むこと**が目的です。

---

# 🧪 今日の重要概念

今日の内容を一本にすると、

```text
データ
 ↓
train/test分割
 ↓
Pipeline
 ↓
GridSearchCV
 ↓
K-Fold Cross Validation
 ↓
ハイパーパラメータ比較
 ↓
最適設定
 ↓
testで最終評価
```

となります。

そして、

```text
モデルパラメータ
    ↓
データから学習

ハイパーパラメータ
    ↓
人間が設定
    ↓
CVなどで選択
```

という区別も覚えておきましょう。

---

# 🌱 第19回のまとめ

今日の核心はこれです。

> **機械学習では、モデルそのものだけでなく「モデルをどう設定するか」も実験対象になる。**

そして、

> **ハイパーパラメータを選ぶときは、交差検証を使い、最終テストデータは最後まで温存する。**

これで、

```text
前処理
 ↓
特徴量
 ↓
モデル
 ↓
評価
 ↓
Cross Validation
 ↓
ハイパーパラメータ探索
```

という、かなり本格的なscikit-learn開発の骨格ができてきました。

---

# 🔜 第20回予告

## 汎化・過学習・正則化：なぜ「訓練データで100点」が危険なのか

次回は、ここまでの話を支える**機械学習の核心概念**に入ります。

* 汎化（generalization）
* 過学習（overfitting）
* 未学習（underfitting）
* 正則化（regularization）
* Bias–Varianceの考え方
* `C` と正則化の関係

を扱います。

ここで初めて、

> **「モデルがデータを覚えること」と「未知のデータに対応できること」は違う**

という機械学習の本質を、コードと実験結果の両方から理解します。

そしてこの先の**決定木・アンサンブル学習・PyTorch**へつながる重要回です。